# धडा 01 - AI एजंटचा परिचय

**AI एजंट्स फॉर बिगिनर्स** कोर्समधील पहिल्या धड्यात आपले स्वागत आहे!

एक **AI एजंट** म्हणजे एक प्रोग्राम जो मोठा भाषा मॉडेल (LLM) त्याच्या तर्कशक्तीसाठी वापरतो आणि प्रत्यक्ष जगात *क्रिया* करू शकतो — API कॉल करणे, डेटाबेस क्वेरी करणे, किंवा कोड चालवणे — जेणेकरून वापरकर्त्यासाठी एखादा उद्दिष्ट साध्य करू शकेल.

या नोटबुकमध्ये तुम्ही तुमचा पहिला एजंट तयार कराल: एक **ट्रॅव्हल एजंट** जो सुट्टीसाठी ठिकाणे सुचवतो. मार्गक्रमे तुम्ही शिकणार आहात:

1. **Microsoft Agent Framework** वापरून Microsoft Foundry Agent Service शी कनेक्ट होणे.
2. एजंटला एक **टूल** देणे — एक सोपी Python फंक्शन जी तो कॉल करू शकतो.
3. एजंट चालवणे आणि त्याचा प्रतिसाद तपासणे.
4. एजंटचा प्रतिसाद टोकननिहाय प्रवाहित करणे.


## सेटअप

या नोटबुक चालवण्यापूर्वी, खात्री करा की:

1. **एक Microsoft Foundry प्रकल्प** ज्यामध्ये एक चॅट मॉडेल डिप्लॉय केलेले आहे (उदा. `gpt-4.1-mini`).
2. **Azure CLI मध्ये लॉगिन केलेले आहे** — तुमच्या टर्मिनलमध्ये `az login` चालवा.
3. **आवश्यक एन्व्हायर्नमेंट व्हेरिएबल सेट केले आहेत:**
   - `AZURE_AI_PROJECT_ENDPOINT` — तुमच्या Microsoft Foundry प्रकल्पाचा एंडपॉइंट.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तुमच्या डिप्लॉय केलेल्या मॉडेलचे नाव.

खालील सेलमध्ये तुम्हाला आवश्यक असलेले Python पॅकेजेस इन्स्टॉल होतात.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## तुमचा पहिला एजंट तयार करणे

एजन्टला दोन गोष्टी आवश्यक असतात:

- **सूचना** ज्या त्याला *तो कोण आहे* आणि *कसा वागायचा* हे सांगतात (एक सिस्टम प्रॉम्प्ट).
- **साधने** — `@tool` ने सजवलेल्या Python फंक्शन्स जे एजंट माहिती मिळवण्यासाठी किंवा क्रिया करण्यासाठी कॉल करू शकतो.

खाली आपण एक सोपी साधन परिभाषित केली आहे जी लोकप्रिय सुट्टीसाठीच्या गंतव्यांची यादी परत करते. जेव्हा वापरकर्ता प्रवासाच्या शिफारसी विचारतो तेव्हा एजंट हे साधन वापरेल.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रीमिंग प्रतिसाद

अधिक संवादात्मक अनुभवासाठी तुम्ही एजंटच्या प्रतिसादाचे **स्ट्रीम** करू शकता. पूर्ण उत्तराची प्रतीक्षा करण्याऐवजी, एजंट तयार करताना मजकूराच्या तुकड्यांना देते. हे विशेषतः चॅट इंटरफेस मध्ये उपयुक्त आहे जिथे तुम्हाला आउटपुट रिअल टाईममध्ये दाखवायचे असते.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

या धड्यात तुम्ही कसे शिकले:

- **एक प्रदाता तयार करा** जो Microsoft Foundry Agent Service शी `FoundryChatClient` द्वारे कनेक्ट होईल.
- **`@tool` डेकोरेटर वापरून टूल परिभाषित करा जेणेकरून एजंट तुमच्या Python फंक्शन्सना कॉल करू शकेल.
- **एजंट वापरून चालवा** वापरकर्त्याचा संदेश देऊन आणि त्याचे प्रतिसाद प्रिंट करा.
- **रेस्पॉन्सेस स्ट्रीम करा** realtime आउटपुटसाठी.

पुढील धड्यात आपण एजंटिक फ्रेमवर्क्स अधिक सखोलपणे पाहणार आहोत आणि एजंट्सना अधिक सामर्थ्यवान टूल्स आणि मल्टी-स्टेप विचार क्षमताही कशा दिल्या जातात हे शिकणार आहोत.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
